# Ask the Constitution — Pakistan
RAG over the Constitution of Pakistan: crawl articles, build a MiniLM+FAISS index, evaluate retrieval, demo Q&A.

In [1]:
!pip -q install sentence-transformers faiss-cpu beautifulsoup4 requests

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try apt install
    python3-xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a non-Debian-packaged Python package,
    create a virtual environment using python3 -m venv path/to/venv.
    Then use path/to/venv/bin/python and path/to/venv/bin/pip. Make
    sure you have python3-full installed.
    
    If you wish to install a non-Debian packaged Python application,
    it may be easiest to use pipx install xyz, which will manage a
    virtual environment for you. Make sure you have pipx installed.
    
    See /usr/share/doc/python3.12/README.venv for more information.

note: If you believe this is a mistake, please contact your Python installation or OS distribution provider. You can override this, at the risk of breaking your Python installation or OS, by passing --break-system-packages.
hint: See PEP 668 for the detai

In [2]:
import json, re, time, requests
from bs4 import BeautifulSoup
print('imports ok')

imports ok


## 1. Crawl the constitution (323 articles + 5 schedules, pakistani.org)

In [3]:
import json, re, time
import requests
from bs4 import BeautifulSoup

BASE = "https://pakistani.org"
H = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36"}

def get(path):
    r = requests.get(BASE + path, headers=H, timeout=30)
    r.raise_for_status()
    return r.text

ART_RE = re.compile(r"^(\d{1,3}[A-Z]?)\.?$")

def parse_part(path):
    soup = BeautifulSoup(get(path), "html.parser")
    articles = []
    for tr in soup.find_all("tr"):
        tds = tr.find_all("td", recursive=False)
        if len(tds) != 2:
            continue
        nobr = tds[0].find("nobr")
        if not nobr:
            continue
        b = nobr.find("b")
        m = ART_RE.match((b or nobr).get_text(strip=True))
        if not m:
            continue
        no = m.group(1)
        # second td: <b>Title</b><br> body...
        for sup in tds[1].find_all("sup"):  # drop footnote markers
            sup.decompose()
        title_tag = tds[1].find("b")
        title = title_tag.get_text(strip=True) if title_tag else ""
        if title_tag:
            title_tag.decompose()
        body = tds[1].get_text("\n", strip=True)
        if body:
            articles.append({"article": no, "title": title, "text": body, "source": BASE + path})
    return articles

def clean_lines(html):
    soup = BeautifulSoup(html, "html.parser")
    for t in soup(["script", "style"]):
        t.decompose()
    text = soup.get_text("\n")
    return [l.strip() for l in text.split("\n") if l.strip()]


def parse_schedule(path, name, subtitle):
    lines = clean_lines(get(path))
    start = 0
    for j, l in enumerate(lines):
        if l == name:
            start = j
            break
    body = []
    for l in lines[start:]:
        if l in ("Go to:", "The Constitution of Pakistan"):
            break
        body.append(l)
    text = "\n".join(body)
    text = text.replace(name, "", 1).replace(subtitle, "", 1).strip()
    return {"article": name, "title": subtitle, "text": text, "source": BASE + path}


def main():
    idx = get("/pakistan/constitution/")
    soup = BeautifulSoup(idx, "html.parser")
    part_paths = sorted({a["href"] for a in soup.find_all("a", href=True)
                         if re.match(r"^/pakistan/constitution/part\d+(\.ch\d+[A-Z]?)?\.html$", a["href"] or "")})
    sched_paths = sorted({a["href"] for a in soup.find_all("a", href=True)
                          if re.match(r"^/pakistan/constitution/schedules/schedule[1-5]\.html$", a["href"] or "")})
    print("part pages:", len(part_paths), "| schedules:", len(sched_paths))
    all_articles, seen = [], set()
    for p in part_paths:
        arts = parse_part(p)
        print(p, "->", len(arts))
        for a in arts:
            if a["article"] in seen:
                continue
            seen.add(a["article"])
            all_articles.append(a)
        time.sleep(0.4)
    sched_info = [
        ("First Schedule", "Laws exempted from the operation of Article 8(1) and (2)"),
        ("Second Schedule", "Election of President"),
        ("Third Schedule", "Oaths of Office"),
        ("Fourth Schedule", "Legislative Lists"),
        ("Fifth Schedule", "Remuneration and Terms and Conditions of Service of Judges"),
    ]
    for (name, subtitle), p in zip(sched_info, sched_paths):
        s = parse_schedule(p, name, subtitle)
        print(p, "->", len(s["text"]), "chars")
        all_articles.append(s)
        time.sleep(0.4)

    def key(a):
        m = re.match(r"(\d+)([A-Z]?)", a["article"])
        if m:
            return (0, int(m.group(1)), m.group(2))
        return (1, 0, a["article"])
    all_articles.sort(key=key)
    json.dump(all_articles, open("articles.json", "w"), indent=1, ensure_ascii=False)
    print("TOTAL:", len(all_articles), "| first:", all_articles[0]["article"], "| last:", all_articles[-1]["article"])
    have = {a["article"] for a in all_articles}
    missing = [str(i) for i in range(1, 281) if str(i) not in have]
    print("missing plain 1-280:", missing)
    empties = [a["article"] for a in all_articles if not a["text"] or len(a["text"]) < 20]
    print("suspiciously short:", empties)

main()


part pages: 33 | schedules: 5


/pakistan/constitution/part1.html -> 7


/pakistan/constitution/part10.html -> 6


/pakistan/constitution/part11.html -> 4


/pakistan/constitution/part12.ch1.html -> 3


...

/pakistan/constitution/schedules/schedule2.html -> 8507 chars


/pakistan/constitution/schedules/schedule3.html -> 23260 chars


/pakistan/constitution/schedules/schedule4.html -> 18186 chars


/pakistan/constitution/schedules/schedule5.html -> 6326 chars


TOTAL: 328 | first: 1 | last: Third Schedule
missing plain 1-280: []
suspiciously short: []


## 2. Chunk, embed (MiniLM), index (FAISS)

In [4]:
"""Build retrieval index: article-aware chunks + MiniLM embeddings + FAISS."""
import json, re
import numpy as np

CHUNK = 800
OVERLAP = 120

def split(text):
    if len(text) <= CHUNK:
        return [text]
    parts, i = [], 0
    while i < len(text):
        j = min(i + CHUNK, len(text))
        if j < len(text):  # break at sentence end when possible
            k = text.rfind(". ", i + CHUNK - 200, j)
            if k > i + 200:
                j = k + 1
        parts.append(text[i:j].strip())
        i = j - OVERLAP if j < len(text) else j
    return [p for p in parts if p]

def main():
    arts = json.load(open("articles.json"))
    chunks = []
    for a in arts:
        header = f"Article {a['article']}: {a['title']}"
        for k, part in enumerate(split(a["text"])):
            chunks.append({
                "article": a["article"],
                "title": a["title"],
                "part": k,
                "n_parts": 0,  # filled below
                "text": f"{header}\n{part}",
                "source": a["source"],
            })
    # fill n_parts
    from collections import Counter
    counts = Counter((c["article"]) for c in chunks)
    for c in chunks:
        c["n_parts"] = counts[c["article"]]

    from sentence_transformers import SentenceTransformer
    model = SentenceTransformer("all-MiniLM-L6-v2")
    texts = [c["text"] for c in chunks]
    emb = model.encode(texts, batch_size=32, show_progress_bar=True, normalize_embeddings=True)

    import faiss
    index = faiss.IndexFlatIP(emb.shape[1])
    index.add(np.asarray(emb, dtype=np.float32))
    faiss.write_index(index, "faiss.index")
    json.dump(chunks, open("chunks.json", "w"), ensure_ascii=False, indent=1)
    np.save("embeddings.npy", emb)
    print(f"chunks={len(chunks)} dim={emb.shape[1]}")
    # coverage: every article has >=1 chunk, every chunk non-empty
    assert len({c["article"] for c in chunks}) == len(arts), "coverage broken"
    assert all(len(c["text"]) > 30 for c in chunks), "empty chunk found"
    print("coverage OK:", len(arts), "articles ->", len(chunks), "chunks")

main()


/home/hatch/workspace/constitution-rag/venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3343.11it/s]

Batches:   0%|          | 0/24 [00:00<?, ?it/s]

Batches:   4%|▍         | 1/24 [00:04<01:36,  4.19s/it]

...

Batches:  92%|█████████▏| 22/24 [00:59<00:03,  1.54s/it]

Batches:  96%|█████████▌| 23/24 [00:59<00:01,  1.28s/it]

Batches: 100%|██████████| 24/24 [01:00<00:00,  1.05s/it]

Batches: 100%|██████████| 24/24 [01:00<00:00,  2.52s/it]

chunks=768 dim=384
coverage OK: 328 articles -> 768 chunks


## 3. Evaluate retrieval on 30 gold questions

In [5]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

EVAL = [
    ("What does the constitution say about freedom of speech?", ["19"]),
    ("Is there a right to a fair trial?", ["10A"]),
    ("Does the constitution guarantee free education for children?", ["25A"]),
    ("What is the right to information?", ["19A"]),
    ("Are all citizens equal before the law?", ["25"]),
    ("What protections exist against arbitrary arrest and detention?", ["10"]),
    ("What does the constitution say about freedom of religion?", ["20"]),
    ("Can the government discriminate on the basis of sex?", ["25", "27"]),
    ("What are the qualifications to become a member of parliament?", ["62"]),
    ("On what grounds can a parliament member be disqualified?", ["63"]),
    ("How is the President of Pakistan elected?", ["41"]),
    ("What is the role of the Council of Common Interests?", ["153"]),
    ("How can the constitution be amended?", ["238", "239"]),
    ("When can emergency be proclaimed in Pakistan?", ["232"]),
    ("What is the original jurisdiction of the Supreme Court?", ["184"]),
    ("What powers does a High Court have to enforce fundamental rights?", ["199"]),
    ("Who commands the Armed Forces of Pakistan?", ["243"]),
    ("What does the constitution say about the dignity of man?", ["14"]),
    ("Is there freedom of association and forming political parties?", ["17"]),
    ("How are seats in the National Assembly distributed?", ["51"]),
    ("What is the National Finance Commission?", ["160"]),
    ("How are judges of the superior courts appointed?", ["175A"]),
    ("What is the procedure for passing bills in parliament?", ["70"]),
    ("Who is the Chief Election Commissioner and how are they appointed?", ["213"]),
    ("When must general elections be held?", ["224"]),
    ("What do the Islamic provisions require of laws in Pakistan?", ["227", "230", "231"]),
    ("What is the Objectives Resolution and where is it in the constitution?", ["2A"]),
    ("What are the Principles of Policy?", ["29", "30"]),
    ("Can fundamental rights be suspended?", ["233", "234"]),
    ("What is the Federal Shariat Court?", ["203A", "203B", "203C", "203D", "203E", "203F", "203G", "203H", "203J"]),
]


def main():
    chunks = json.load(open("chunks.json"))
    index = faiss.read_index("faiss.index")
    model = SentenceTransformer("all-MiniLM-L6-v2")

    questions = [q for q, _ in EVAL]
    q_emb = model.encode(questions, normalize_embeddings=True, show_progress_bar=False)
    D, I = index.search(np.asarray(q_emb, dtype=np.float32), 5)

    hits1 = hits3 = hits5 = 0
    misses = []
    for (q, gold), idxs in zip(EVAL, I):
        retrieved = [chunks[i]["article"] for i in idxs]
        # a hit if ANY gold article appears in top-k (chunk-level -> article-level)
        if any(g in retrieved[:1] for g in gold):
            hits1 += 1
        if any(g in retrieved[:3] for g in gold):
            hits3 += 1
        if any(g in retrieved[:5] for g in gold):
            hits5 += 1
        else:
            misses.append((q, gold, retrieved))
    n = len(EVAL)
    print(f"n={n}")
    print(f"hit-rate@1: {hits1/n:.3f} ({hits1}/{n})")
    print(f"hit-rate@3: {hits3/n:.3f} ({hits3}/{n})")
    print(f"hit-rate@5: {hits5/n:.3f} ({hits5}/{n})")
    print("\nmisses@5:")
    for q, gold, ret in misses:
        print(f"  Q: {q}\n    gold={gold} retrieved={ret}")
    json.dump({"n": n, "hr1": hits1/n, "hr3": hits3/n, "hr5": hits5/n,
               "misses": [{"q": q, "gold": gold, "retrieved": ret} for q, gold, ret in misses]},
              open("metrics.json", "w"), indent=1)

main()


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 30217.06it/s]

n=30
hit-rate@1: 0.867 (26/30)
hit-rate@3: 0.933 (28/30)
hit-rate@5: 1.000 (30/30)

misses@5:


## 4. Try it: retrieve articles for a question

In [6]:
from rag import retrieve
for r in retrieve('What does the constitution say about freedom of speech?', k=3):
    print(f"Article {r['article']}: {r['title']} (score={r['score']:.3f})")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 22070.77it/s]

Article 19: Freedom of speech, etc. (score=0.627)
Article 20: Freedom to profess religion and to manage religious institutions. (score=0.463)
Article 19A: Right to information: (score=0.460)
